# Single-example activation trajectories before each attention layer

This notebook uses the same seeded example-selection logic as Notebook 19. For the selected example, it collects the residual-stream activation immediately before every attention layer, fits a separate 3D PCA for each layer, and plots the token-position trajectory. It also plots the residual stream after the final attention sublayer and before its MLP. Each 3D plot is paired with a separately fitted 2D PCA plot. For each pre-attention layer, it also shows token-length change versus consecutive-token distance in that layer's 3D PCA space.

Points are colored by the cumulative character count at that token position. Each plot includes token-position hover labels, the origin, and faint vector segments from the origin.

In [33]:
from pathlib import Path
import json

import numpy as np
import plotly.graph_objects as go
import torch
from sklearn.decomposition import PCA
from IPython.display import display

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
RANDOM_SEED = 19_003  # Same seed as Notebook 19.
EXAMPLE_INDEX = None   # None reproduces Notebook 19's seeded random choice.
DISPLAY_2D_PLOTS = False  # Set False to display only the 3D trajectories.
COLLECTION_BATCH_SIZE = 128


In [34]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()

n_layers = cfg["model"]["n_layers"]
activation_sites = [f"blocks.{layer}.hook_resid_pre" for layer in range(n_layers)]
post_final_attention_site = f"blocks.{n_layers - 1}.hook_resid_mid"
cache_sites = activation_sites + [post_final_attention_site]
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Activation sites before attention: {activation_sites}")
print(f"Post-final-attention site: {post_final_attention_site}")


Building vocab...
('BOS', '_NEWLINE_')
Loaded checkpoint step 10000 on cpu
Activation sites before attention: ['blocks.0.hook_resid_pre', 'blocks.1.hook_resid_pre', 'blocks.2.hook_resid_pre']
Post-final-attention site: blocks.2.hook_resid_mid


## Sample the same example as Notebook 19

In [35]:
generator = torch.Generator(device="cpu").manual_seed(RANDOM_SEED)
batch = project_data.make_batch(
    batch_size=COLLECTION_BATCH_SIZE,
    vocab=vocab,
    task_cfg=cfg["task"],
    device=device,
    seed=RANDOM_SEED,
)
if EXAMPLE_INDEX is None:
    example_index = int(torch.randint(COLLECTION_BATCH_SIZE, (), generator=generator).item())
else:
    example_index = int(EXAMPLE_INDEX)
if not 0 <= example_index < COLLECTION_BATCH_SIZE:
    raise ValueError(f"EXAMPLE_INDEX must be between 0 and {COLLECTION_BATCH_SIZE - 1}.")

_, cache = model.run_with_cache(
    batch.tokens,
    prepend_bos=False,
    names_filter=lambda name: name in set(cache_sites),
)
activations_by_layer = {
    layer: cache[site][example_index].detach().cpu().float().numpy()
    for layer, site in enumerate(activation_sites)
}
post_final_attention_activation = cache[post_final_attention_site][example_index].detach().cpu().float().numpy()
tokens = batch.tokens[example_index].detach().cpu().tolist()
targets = batch.targets[example_index].detach().cpu().tolist()
line_width = int(batch.line_lengths[example_index].item())

character_counts = []
running_count = 0
for token_id in tokens:
    if token_id == vocab.newline_id:
        running_count = 0
    else:
        running_count += project_data.extract_character_count(vocab.decode_token(token_id))
    character_counts.append(running_count)

token_names = [vocab.decode_token(token_id) for token_id in tokens]
target_names = [vocab.decode_token(token_id) for token_id in targets]
newline_query_positions = [i for i, target_id in enumerate(targets) if target_id == vocab.newline_id]

print(f"Seed: {RANDOM_SEED}; example index: {example_index}; sampled line width: {line_width}")
print(f"Token positions: {len(tokens)}; target-newline positions: {newline_query_positions}")
for layer, activation in activations_by_layer.items():
    print(f"Layer {layer}: activation shape {activation.shape}")
print(f"After final attention: activation shape {post_final_attention_activation.shape}")


Seed: 19003; example index: 27; sampled line width: 61
Token positions: 150; target-newline positions: [14, 32, 45, 57, 75, 91, 107, 121, 137]
Layer 0: activation shape (150, 128)
Layer 1: activation shape (150, 128)
Layer 2: activation shape (150, 128)
After final attention: activation shape (150, 128)


## 3D PCA trajectory before each attention layer

In [36]:
def display_2d_activation_pca(activations, title):
    pca_2d = PCA(n_components=2)
    coordinates_2d = pca_2d.fit_transform(activations)
    segments_2d = []
    for point in coordinates_2d:
        segments_2d.extend([[0, 0], point.tolist(), [None, None]])

    fig_2d = go.Figure()
    fig_2d.add_trace(go.Scatter(
        x=[0], y=[0], mode="markers", marker=dict(size=9, color="black"),
        name="Origin", hovertemplate="(0, 0)<extra>Origin</extra>",
    ))
    fig_2d.add_trace(go.Scatter(
        x=[point[0] for point in segments_2d],
        y=[point[1] for point in segments_2d],
        mode="lines", line=dict(color="rgba(100,100,100,0.18)", width=1),
        name="Activation vectors", showlegend=False, hoverinfo="skip",
    ))
    fig_2d.add_trace(go.Scatter(
        x=coordinates_2d[:, 0], y=coordinates_2d[:, 1], mode="lines+markers",
        marker=dict(size=6, color=character_counts, colorscale="Viridis",
                    colorbar=dict(title="Character count")),
        line=dict(color="rgba(80,80,80,0.6)", width=3), name="Activation",
        text=[f"{i}: {name}" for i, name in enumerate(token_names)],
        customdata=np.asarray(character_counts),
        hovertemplate="position=%{text}<br>character count=%{customdata}<extra>Activation</extra>",
    ))
    fig_2d.update_layout(
        title=title + f"<br>2D PCA variance explained = {pca_2d.explained_variance_ratio_.sum():.1%}",
        xaxis_title="PC1", yaxis_title="PC2", height=700,
        margin=dict(l=60, r=40, b=60, t=90),
    )
    display(fig_2d)

for layer, activations in activations_by_layer.items():
    pca = PCA(n_components=3)
    coordinates = pca.fit_transform(activations)
    segments = []
    for point in coordinates:
        segments.extend([[0, 0, 0], point.tolist(), [None, None, None]])

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0], mode="markers",
        marker=dict(size=7, color="black"), name="Origin",
        hovertemplate="(0, 0, 0)<extra>Origin</extra>",
    ))
    fig.add_trace(go.Scatter3d(
        x=[p[0] for p in segments], y=[p[1] for p in segments], z=[p[2] for p in segments],
        mode="lines", line=dict(color="rgba(100,100,100,0.18)", width=1),
        name="Activation vectors", showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter3d(
        x=coordinates[:, 0], y=coordinates[:, 1], z=coordinates[:, 2],
        mode="lines+markers",
        marker=dict(size=5, color=character_counts, colorscale="Viridis",
                    colorbar=dict(title="Character count")),
        line=dict(color="rgba(80,80,80,0.6)", width=3),
        name="Activation",
        text=[f"{i}: {name}" for i, name in enumerate(token_names)],
        customdata=np.asarray(character_counts),
        hovertemplate="position=%{text}<br>character count=%{customdata}<extra>Activation</extra>",
    ))
    fig.update_layout(
        title=(f"Example {example_index}: activation before attention layer {layer}<br>"
               f"PCA variance explained = {pca.explained_variance_ratio_.sum():.1%}"),
        scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
        height=800, margin=dict(l=0, r=0, b=0, t=90),
    )
    display(fig)

    # Compare consecutive non-newline tokens in this layer's 3D PCA space.
    # The character-count difference equals the later token's character length.
    pair_positions = []
    token_length_differences = []
    pca_distances = []
    full_embedding_distances = []
    pair_labels = []
    for position in range(len(tokens) - 1):
        if tokens[position] == vocab.newline_id or tokens[position + 1] == vocab.newline_id:
            continue
        later_token_length = (
            character_counts[position + 1] - character_counts[position]
        )
        distance = float(np.linalg.norm(
            coordinates[position + 1] - coordinates[position]
        ))
        full_distance = float(np.linalg.norm(
            activations[position + 1] - activations[position]
        ))
        pair_positions.append(position)
        token_length_differences.append(later_token_length)
        pca_distances.append(distance)
        full_embedding_distances.append(full_distance)
        pair_labels.append(
            f"{position}->{position + 1}: {token_names[position]} -> {token_names[position + 1]}"
        )

    fig_pairs = go.Figure()
    fig_pairs.add_trace(go.Scatter(
        x=token_length_differences,
        y=pca_distances,
        mode="markers",
        marker=dict(size=8, color=token_length_differences, colorscale="Viridis",
                    colorbar=dict(title="Later token length")),
        text=pair_labels,
        customdata=np.asarray(pair_positions),
        hovertemplate=(
            "%{text}<br>later token length=%{x}<br>"
            "3D PCA distance=%{y:.4f}<extra>3D PCA distance</extra>"
        ),
        name="3D PCA distance",
    ))
    fig_pairs.add_trace(go.Scatter(
        x=token_length_differences,
        y=full_embedding_distances,
        mode="markers",
        marker=dict(size=8, color="crimson", symbol="x"),
        text=pair_labels,
        customdata=np.asarray(pair_positions),
        hovertemplate=(
            "%{text}<br>later token length=%{x}<br>"
            "full embedding distance=%{y:.4f}<extra>Full embedding distance</extra>"
        ),
        name="Full embedding distance",
    ))
    fig_pairs.update_layout(
        title=f"Layer {layer}: token-length change vs. 3D PCA activation distance",
        xaxis_title="Character-count difference / later token length",
        yaxis_title="Euclidean distance",
        height=650, margin=dict(l=70, r=40, b=70, t=90),
    )
    display(fig_pairs)

    correlation_matrix = np.corrcoef([
        np.asarray(token_length_differences, dtype=float),
        np.asarray(pca_distances, dtype=float),
        np.asarray(full_embedding_distances, dtype=float),
    ])
    print(
        f"Layer {layer} Pearson correlations: "
        f"token length vs 3D PCA distance = {correlation_matrix[0, 1]:+.4f}; "
        f"token length vs full embedding distance = {correlation_matrix[0, 2]:+.4f}; "
        f"3D PCA distance vs full embedding distance = {correlation_matrix[1, 2]:+.4f}"
    )

    if DISPLAY_2D_PLOTS:
        display_2d_activation_pca(
            activations,
            f"Example {example_index}: activation before attention layer {layer}",
        )

# Plot the residual stream immediately after the final attention sublayer
# (and before the final layer's MLP).
activations = post_final_attention_activation
pca = PCA(n_components=3)
coordinates = pca.fit_transform(activations)
segments = []
for point in coordinates:
    segments.extend([[0, 0, 0], point.tolist(), [None, None, None]])
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers", marker=dict(size=7, color="black"),
    name="Origin", hovertemplate="(0, 0, 0)<extra>Origin</extra>",
))
fig.add_trace(go.Scatter3d(
    x=[p[0] for p in segments], y=[p[1] for p in segments], z=[p[2] for p in segments],
    mode="lines", line=dict(color="rgba(100,100,100,0.18)", width=1),
    name="Activation vectors", showlegend=False, hoverinfo="skip",
))
fig.add_trace(go.Scatter3d(
    x=coordinates[:, 0], y=coordinates[:, 1], z=coordinates[:, 2], mode="lines+markers",
    marker=dict(size=5, color=character_counts, colorscale="Viridis",
                colorbar=dict(title="Character count")),
    line=dict(color="rgba(80,80,80,0.6)", width=3), name="Activation",
    text=[f"{i}: {name}" for i, name in enumerate(token_names)],
    customdata=np.asarray(character_counts),
    hovertemplate="position=%{text}<br>character count=%{customdata}<extra>Activation</extra>",
))
fig.update_layout(
    title=(f"Example {example_index}: activation after final attention layer "
           f"{n_layers - 1} and before its MLP<br>"
           f"PCA variance explained = {pca.explained_variance_ratio_.sum():.1%}"),
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    height=800, margin=dict(l=0, r=0, b=0, t=90),
)
display(fig)

if DISPLAY_2D_PLOTS:
    display_2d_activation_pca(
        activations,
        f"Example {example_index}: activation after final attention layer "
        f"{n_layers - 1} and before its MLP",
    )


Layer 0 Pearson correlations: token length vs 3D PCA distance = +0.0033; token length vs full embedding distance = -0.0076; 3D PCA distance vs full embedding distance = +0.8834


Layer 1 Pearson correlations: token length vs 3D PCA distance = -0.0079; token length vs full embedding distance = -0.0749; 3D PCA distance vs full embedding distance = +0.9246


Layer 2 Pearson correlations: token length vs 3D PCA distance = +0.4744; token length vs full embedding distance = +0.4460; 3D PCA distance vs full embedding distance = +0.9223
